In [1]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
print("전체:", len(ds))
print("type :", Counter(ds["type"]))
print("level:", Counter(ds["level"]))
print("type×level:", Counter(zip(ds["type"], ds["level"])))

전체: 7405
type : Counter({'bridge': 5918, 'comparison': 1487})
level: Counter({'hard': 7405})
type×level: Counter({('bridge', 'hard'): 5918, ('comparison', 'hard'): 1487})


In [3]:
import random
from collections import defaultdict
random.seed(42)
SUBSET_SIZE = 500

groups = defaultdict(list)
for i, (t,l) in enumerate(zip(ds["type"],ds["level"])):
    groups[(t,l)].append(i)
    
picked = []
for key,idxs in groups.items():
    k = round(len(idxs)/len(ds)*SUBSET_SIZE)
    picked += random.sample(idxs,min(k,len(idxs)))
    
random.shuffle(picked)
subset = ds.select(picked)
print("서브셋:",len(subset))
print("서브셋 typexlevel: ", Counter(zip(subset["type"], subset["level"])))

서브셋: 500
서브셋 typexlevel:  Counter({('bridge', 'hard'): 400, ('comparison', 'hard'): 100})


In [5]:
corpus = {}   # title -> 문단 텍스트
for ex in subset:
    for t, s in zip(ex["context"]["title"], ex["context"]["sentences"]):
        if t not in corpus:
            corpus[t] = " ".join(s)   # 어제 확인한 문단 복원법
print("고유 문단 수:", len(corpus))   # 500×10=5000 에서 중복 제거된 값

고유 문단 수: 4928


In [7]:
import json, os
os.makedirs("../data", exist_ok=True)

json.dump(corpus, open("../data/corpus.json", "w", encoding="utf-8"),
          ensure_ascii=False)

qa = [{"question": ex["question"], "answer": ex["answer"],
       "gold_titles": ex["supporting_facts"]["title"],
       "type": ex["type"], "level": ex["level"]} for ex in subset]
json.dump(qa, open("../data/qa.json", "w", encoding="utf-8"),
          ensure_ascii=False)
print("저장 완료:", len(corpus), "문단 /", len(qa), "질문")

저장 완료: 4928 문단 / 500 질문
